# Data Preprocessing for SMS Phishing 

In [9]:
import os

import pandas as pd
import re
import warnings
from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split

In [10]:
warnings.filterwarnings('ignore', category=UserWarning, module='bs4')

# 1. Load datasets
#real_csv_path = '/content/drive/MyDrive/Colab Notebooks/7007Project/sms-spam-collection.csv'
real_csv_path = r'C:\\Users\\Ong Hui Ling\\Dropbox\\PC\\Documents\\Github\\SMS-Phishing-Detection\\Dataset\\Raw\\sms_real_dataset.csv'
#real_csv_path = "/content/drive/MyDrive/SMS-Phishing-Detection/Dataset/Raw/sms_real_dataset.csv"

#llm_csv_path = '/content/drive/MyDrive/Colab Notebooks/7007Project/Dataset_10191.csv'
llm_csv_path = r'C:\\Users\\Ong Hui Ling\\Dropbox\\PC\\Documents\\Github\\SMS-Phishing-Detection\\Dataset\\Raw\\llm_augmented_dataset.csv'
#llm_csv_path = "/content/drive/MyDrive/SMS-Phishing-Detection/Dataset/Raw/llm_augmented_dataset.csv"

real_df = pd.read_csv(real_csv_path)
llm_df = pd.read_csv(llm_csv_path)

# 1.1 Standardize column names 
for df in [real_df, llm_df]:
    if 'v1' in df.columns and 'v2' in df.columns:
        df.rename(columns={'v1': 'label', 'v2': 'text'}, inplace=True)
    elif 'text' not in df.columns:
        df.columns = ['label', 'text'] + list(df.columns[2:])

# 1.2 Keep only essential columns
real_df = real_df[['label', 'text']].dropna()
llm_df = llm_df[['label', 'text']].dropna()

# 1.3 Normalise label casing/spacing
for df in [real_df, llm_df]:
    df['label'] = df['label'].astype(str).str.strip().str.lower()

print("=== Real dataset label distribution (before split) ===")
real_counts = real_df['label'].value_counts()
for label, cnt in real_counts.items():
    print(f"  {label:>10}: {cnt:5d} ({cnt/len(real_df)*100:.1f}%)")

# 2. Basic Text Cleaning & Regex Entity Masking
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    # 2.1 Lowercasing
    text = text.lower()

    # 2.2 Remove HTML tags
    text = BeautifulSoup(text, "html.parser").get_text()

    # 2.3 Remove meaningless Unicode (keep basic ASCII)
    text = text.encode('ascii', 'ignore').decode('utf-8')

    # 2.4 Regex Entity Masking: Replace URLs with <URL>
    url_pattern = re.compile(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+')
    text = url_pattern.sub('<URL>', text)

    # 2.5 Basic whitespace cleanup
    text = ' '.join(text.split())
    return text

print("Applying text cleaning...")
real_df['text'] = real_df['text'].apply(preprocess_text)
llm_df['text'] = llm_df['text'].apply(preprocess_text)

# 3. Stratified Split (80/20 on real data)
print("Splitting real data (Stratified)...")
X_train, X_test, y_train, y_test = train_test_split(
    real_df['text'], real_df['label'],
    test_size=0.2,
    stratify=real_df['label'],
    random_state=42
)

train_real_df = pd.DataFrame({'label': y_train, 'text': X_train})
print(f"Real Train Dataset size: {len(train_real_df)}")
test_real_df = pd.DataFrame({'label': y_test, 'text': X_test})
print(f"Real Test Dataset size: {len(test_real_df)}")

print("=== Test dataset label distribution  ===")
real_counts = test_real_df['label'].value_counts()
for label, cnt in real_counts.items():
    print(f"  {label:>10}: {cnt:5d} ({cnt/len(test_real_df)*100:.1f}%)")

# 4. Targeted Overlap Filter
print(f"LLM Dataset size before filtering: {len(llm_df)}")
test_set_texts = set(test_real_df['text'].tolist())
# Filter out rows from LLM data where the text exists in the test set
llm_df_filtered = llm_df[~llm_df['text'].isin(test_set_texts)]
print(f"LLM Dataset size after filtering overlap: {len(llm_df_filtered)}")

# 5. Combine Data
print("Combining training datasets...")
combined_train_df = pd.concat([train_real_df, llm_df_filtered], ignore_index=True)

# 6. Drop duplicates
print(f"Combined Train Dataset size before deduplication: {len(combined_train_df)}")
combined_train_df = combined_train_df.drop_duplicates(subset=['text'])
print(f"Combined Train Dataset size after deduplication: {len(combined_train_df)}")

# 7. Export
#train_export_path = "/content/drive/MyDrive/SMS-Phishing-Detection/Dataset/Processed/train_dataset.csv"
train_export_path = r'C:\\Users\\Ong Hui Ling\\Dropbox\\PC\\Documents\\Github\\SMS-Phishing-Detection\\Dataset\\Processed\\train_dataset.csv'
#test_export_path = "/content/drive/MyDrive/SMS-Phishing-Detection/Dataset/Processed/test_dataset.csv"
test_export_path = r'C:\\Users\\Ong Hui Ling\\Dropbox\\PC\\Documents\\Github\\SMS-Phishing-Detection\\Dataset\\Processed\\test_dataset.csv'

combined_train_df.to_csv(train_export_path, index=False)
test_real_df.to_csv(test_export_path, index=False)


=== Real dataset label distribution (before split) ===
         ham:  4844 (81.1%)
    smishing:   638 (10.7%)
        spam:   489 (8.2%)
Applying text cleaning...


C:\Users\Ong Hui Ling\AppData\Local\Temp\ipykernel_5812\2534227053.py:43: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  text = BeautifulSoup(text, "html.parser").get_text()


Splitting real data (Stratified)...
Real Train Dataset size: 4776
Real Test Dataset size: 1195
=== Test dataset label distribution  ===
         ham:   969 (81.1%)
    smishing:   128 (10.7%)
        spam:    98 (8.2%)
LLM Dataset size before filtering: 10191
LLM Dataset size after filtering overlap: 8820
Combining training datasets...
Combined Train Dataset size before deduplication: 13596
Combined Train Dataset size after deduplication: 7988
